# Part 5 – Production Analytics Architecture

## Objective

Design a production-ready analytics architecture for collections performance.

The architecture follows:

Raw → Staging → Clean → Golden → Feature → Metrics → Dashboard

In [3]:
layers = [
    "Raw",
    "Staging",
    "Clean",
    "Golden",
    "Feature",
    "Metrics",
    "Dashboard"
]

for i, layer in enumerate(layers, 1):
    print(f"{i}. {layer}")

1. Raw
2. Staging
3. Clean
4. Golden
5. Feature
6. Metrics
7. Dashboard


## 5.1 Architecture Layers

| Layer | Purpose |
|---|---|
| Raw | Stores source data as received from operational systems. |
| Staging | Standardizes schemas, column names, data types, timestamps, and source identifiers. |
| Clean | Applies deduplication, validation, entity resolution, and cleaning rules. |
| Golden | Stores trusted datasets used as the analytical source of truth. |
| Feature | Creates analytical features for customer, account, agent, campaign, and payment analysis. |
| Metrics | Calculates approved business metrics such as recovery amount, recovery rate, and conversion. |
| Dashboard | Presents standardized metrics and trends for business and leadership decisions. |

## 5.2 Data Contracts and Primary Keys

Each production table should have a defined primary key, required fields,
data types, and validation rules.

| Table | Primary Key | Important Fields |
|---|---|---|
| borrowers | borrower_id | phone, email |
| accounts | account_id | borrower_id, opened_at, risk_segment |
| agents | agent_id | agent_name |
| agent_sessions | session_id | agent_id, start_at, end_at |
| campaigns | campaign_id | campaign_name, start_date, end_date |
| daily_targeting | target_id | account_id, campaign_id, target_date |
| calls | call_id | account_id, agent_id, event_at |
| call_attempts | attempt_id | call_id, vendor_id |
| call_dispositions | disposition_id | attempt_id, disposition_code |
| whatsapp_events | event_id | account_id, event_at |
| sms_events | event_id | account_id, event_at |
| field_visits | visit_id | account_id, scheduled_at |
| promises_to_pay | ptp_id | account_id, promised_at |
| payments | payment_id | account_id, event_at, amount |
| vendor_telephony | vendor_id | vendor_name, timezone |
| complaints | complaint_id | account_id, created_at |
| account_status_history | history_id | account_id, event_at, status |

### Data Contract Rules

- Primary keys must be unique and not null.
- Foreign keys must reference valid records in the parent table.
- Event timestamps must use a consistent timezone standard.
- Monetary amounts must use a consistent currency and numeric format.
- Status and disposition values must follow approved code lists.
- Required fields must not contain unexpected null values.
- Schema changes must be versioned and documented.

## 5.3 Data Quality Checks

The production pipeline should run automated data quality checks before data
moves into the Golden layer.

| Check | Rule | Action |
|---|---|---|
| Primary key | Must be unique and not null | Reject or quarantine records |
| Foreign key | Must match a valid parent record | Flag invalid records |
| Duplicates | Exact and business-key duplicates should be checked | Deduplicate or quarantine |
| Missing values | Required fields should not be unexpectedly null | Flag and monitor |
| Timestamps | Valid format and consistent timezone | Correct or quarantine |
| Amounts | Payment amounts must be valid numeric values | Reject invalid values |
| Status values | Must follow approved status codes | Flag invalid values |
| Row counts | Compare with expected ranges | Trigger an alert |

### Monitoring and Anomaly Detection

The production pipeline should monitor:

- Daily row counts by source and table.
- Duplicate rates and missing-value rates.
- Payment amount and payment volume anomalies.
- Sudden changes in recovery rate.
- Unusual changes in campaign or channel mix.
- Failed pipeline jobs and delayed data loads.
- Schema changes and unexpected new values.

Alerts should be generated when important metrics move outside
predefined thresholds.

## 5.4 Incremental Processing

The production pipeline should process only new or changed records during
normal daily runs instead of rebuilding all historical data.

### Incremental Strategy

- Use event timestamps and ingestion timestamps to identify new records.
- Maintain a watermark for each source table.
- Process records received after the previous successful watermark.
- Use an overlap window to capture late-arriving records.
- Reprocess affected dates when corrections are received.
- Keep processing logs for successful and failed pipeline runs.

### Late-Arriving Data

Late-arriving events should be accepted when their event time belongs to an
earlier business period.

The pipeline should:

1. Identify the event using its event timestamp.
2. Load the record into the appropriate historical partition.
3. Recalculate affected Golden, Feature, and Metrics data.
4. Record the correction in the pipeline audit log.

### Backfill Strategy

Backfills should be used when source data is corrected, historical records
arrive late, or a transformation rule changes.

A backfill should:

- Identify the affected date range.
- Reprocess the required source records.
- Re-run downstream transformations.
- Recalculate affected metrics.
- Validate row counts and key business metrics.
- Record the backfill reason, execution time, and affected tables.

### Backfill Strategy

Backfills should be used when source data is corrected, historical records
arrive late, or a transformation rule changes.

A backfill should:

- Identify the affected date range.
- Reprocess the required source records.
- Re-run downstream transformations.
- Recalculate affected metrics.
- Validate row counts and key business metrics.
- Record the backfill reason, execution time, and affected tables.

## 5.5 Data Lineage

Data lineage should track how data moves from source systems to business
metrics.

Example lineage:

Source Systems
↓
Raw Layer
↓
Staging Layer
↓
Clean Layer
↓
Golden Dataset
↓
Feature Layer
↓
Metrics Layer
↓
Dashboard

Important lineage information should include:

- Source table and source system.
- Transformation applied at each layer.
- Primary and foreign key relationships.
- Data quality checks performed.
- Metric calculation logic.
- Last successful processing time.
- Any corrections or backfills applied.

## 5.6 Standard Metric Definitions

| Metric | Definition |
|---|---|
| Recovery Amount | Sum of successful payment amounts for the selected period. |
| Successful Payments | Count of payment records with payment_status = SUCCESS. |
| Paying Accounts | Number of unique accounts with at least one successful payment. |
| Recovery Rate | Paying accounts divided by the defined eligible account population. |
| Payment Conversion | Accounts with a successful payment divided by the defined contacted or targeted population. |
| MoM Recovery Growth | Percentage change in recovery amount compared with the previous month. |
| Average Payment Amount | Total successful recovery amount divided by successful payments. |

All metrics should use the same Golden Dataset and approved denominator definitions
across reports and dashboards.

## 5.7 Production Architecture Summary

The proposed architecture creates a controlled path from raw operational data
to business reporting.

Raw and Staging preserve and standardize source data. Clean and Golden apply
data quality and business rules. Feature and Metrics layers create reusable
analytical outputs. The Dashboard layer exposes only approved metrics.

This structure reduces duplicate logic, improves metric consistency, and makes
data issues easier to trace and correct.

## 5.8 Production Analytics Architecture Diagram

### Main Data Flow

```text
+----------------------+
|  Operational Sources |
+----------+-----------+
           |
           v
+----------------------+
|         RAW          |
|     Source Data      |
+----------+-----------+
           |
           v
+----------------------+
|       STAGING        |
|   Standardization    |
+----------+-----------+
           |
           v
+----------------------+
|        CLEAN         |
| Validation & Cleaning|
+----------+-----------+
           |
           v
+----------------------+
|       GOLDEN         |
|     Trusted Data     |
+----------+-----------+
           |
           v
+----------------------+
|       FEATURE        |
| Analytical Features  |
+----------+-----------+
           |
           v
+----------------------+
|       METRICS        |
|    Business KPIs     |
+----------+-----------+
           |
           v
+----------------------+
|      DASHBOARD       |
|  Business Reporting  |
+----------------------+

### Supporting Controls

```text
Data Quality Checks
        |
        v
Validation & Alerts
        |
        v
Monitoring & Audit Logs

Late-Arriving Data
        |
        v
Reprocessing
        |
        v
Backfills

## 5.9 Part 5 Conclusion

The proposed production architecture provides a controlled and traceable path
from operational data to business reporting.

It addresses data quality, deduplication, schema changes, timestamp handling,
late-arriving data, backfills, metric consistency, lineage, and monitoring.

The Golden Dataset remains the source of truth for analytical reporting, while
the Metrics layer ensures that business KPIs use standardized definitions.

This architecture can support reliable collections analytics and future
scaling of the reporting platform.